# Explainable AI with SHAP
## Explainable Fake News Detection using Transformer-based Language Models

**Student Name:** Abdullah  
**Reg Number:** M24F0044DS009  
**Course:** Natural Language Processing  

### Objective
The objective of this notebook is to apply SHAP (SHapley Additive exPlanations) to interpret the predictions of the fine-tuned BERT model and identify important words contributing to fake news detection.


In [ ]:
# Import Liberaries

import shap
import torch
import numpy as np
import pandas as pd

from transformers import BertTokenizer, BertForSequenceClassification


In [ ]:
# Check device GPU/CPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Load Test Dataset

test_df = pd.read_csv("../data/shorttextpreprocessedtest.csv")
test_df.head()


In [ ]:
# Load Trained Model & Tokenizer

model_path = "../models/bert_fake_news_model"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


In [ ]:
# Define Prediction Function for SHAP

def predict_proba(texts):
    # SHAP kabhi numpy array deta hai
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()

    # make string if not already
    texts = [str(t) for t in texts]

    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    return probs.cpu().numpy()



In [ ]:
# Intialize SHAP Text Explainer

explainer = shap.Explainer(
    predict_proba,
    masker=shap.maskers.Text(tokenizer)
)


In [ ]:
# Select Sample Texts for Explanation

test_df = test_df[test_df['text'].notna()]
test_df['text'] = test_df['text'].astype(str)

sample_texts = test_df['text'].sample(3, random_state=42).tolist()
sample_texts


In [ ]:
# Generate SHAP Values

shap_values = explainer(sample_texts)


In [ ]:
# Visualize SHAP Explanations

shap.plots.text(shap_values)


In [ ]:
# Explain prediction for a single instance
shap.plots.text(shap_values[0])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
shap.plots.text(shap_values)
plt.savefig("../results/shap_explanation.png", dpi=300, bbox_inches="tight")
plt.close()


## Interpretation of SHAP Results

- SHAP highlights the most influential words contributing to model predictions.
- Sensational, emotionally charged, or misleading words often push predictions toward fake news.
- Neutral and factual terms contribute positively toward real news classification.
- This interpretability improves trust and transparency in automated fake news detection systems.


## Conclusion

SHAP successfully provides both global and local explanations for BERT-based fake news detection. 
The results demonstrate that the model’s predictions are interpretable and aligned with linguistic patterns commonly observed in fake and real news.
This fulfills the explainability objective of the project.
